In [1]:
import pandas as pd

In [8]:
general_main = '/Users/anzony.quisperojas/Library/CloudStorage/Dropbox/sa_fires'
int_path = f'{general_main}/proj_bureaucrats_farms/data_output/intermediate'

In [61]:
# Winner candidates' name clean
winners = pd.read_csv(f'{general_main}/data/input/my_neta/2008_onwards_winners_table.csv')
winners['name'] = winners.name.str.lower().str.replace(" ","")

# Rubric 1: tweets information
tweets_rubric1 = pd.read_stata(f'{int_path}/tweets_by_rubric1.dta')

In [62]:
tweets_rubric1.columns.tolist()

['index',
 'Politician_Name',
 'year',
 'month',
 'tweets_emotional_mixed',
 'tweets_emotional_negative',
 'tweets_emotional_neutral',
 'tweets_emotional_positive',
 'tweets_emotional_unclear',
 'tweets_agriculture',
 'tweets_development',
 'tweets_election_campaign',
 'tweets_environment_pollution',
 'tweets_farmer_protest',
 'tweets_governance',
 'tweets_party_politics',
 'tweets_sports_culture',
 'tweets_tribute_ceremony',
 'tweets_welfare',
 'number_tweets1_agriculture',
 'number_tweets1_development',
 'number_tweets1_election_campaign',
 'number_tweets1_environment_pollu',
 'number_tweets1_farmer_protest',
 'number_tweets1_governance',
 'number_tweets1_party_politics',
 'number_tweets1_sports_culture',
 'number_tweets1_tribute_ceremony',
 'number_tweets1_unclear',
 'number_tweets1_welfare',
 'tweets_rhetorical_accusatory',
 'tweets_rhetorical_celebratory',
 'tweets_rhetorical_ceremonial',
 'tweets_rhetorical_critical',
 'tweets_rhetorical_grievance',
 'tweets_rhetorical_informatio

In [26]:
# testing the perfect merge between information
left_names  = tweets_rubric1["Politician_Name"]
right_names = winners["name"]


# Unique values on each side
left_unique  = set(left_names.unique())
right_unique = set(right_names.unique())

# Difference checks
only_in_left  = sorted(left_unique - right_unique)   # in tweets, not in winners
only_in_right = sorted(right_unique - left_unique)   # in winners, not in tweets
in_both       = left_unique & right_unique

# Summary
print(f"Unique names in tweets_rubric1 (left):  {len(left_unique):,}")
print(f"Unique names in winners (right):        {len(right_unique):,}")
print(f"In both:                                {len(in_both):,}")
print(f"Only in left (tweets_rubric1):          {len(only_in_left):,}")
print(f"Only in right (winners):                {len(only_in_right):,}")

print("\n--- First 20 names only in LEFT (will get NaN on left-merge) ---")
for n in only_in_left[:20]:
    print(f"  {n}")

print("\n--- First 20 names only in RIGHT (will be dropped on left-merge) ---")
for n in only_in_right[:20]:
    print(f"  {n}")

Unique names in tweets_rubric1 (left):  2,111
Unique names in winners (right):        2,111
In both:                                2,111
Only in left (tweets_rubric1):          0
Only in right (winners):                0

--- First 20 names only in LEFT (will get NaN on left-merge) ---

--- First 20 names only in RIGHT (will be dropped on left-merge) ---


In [47]:
# Panel data in election years
panel_election = pd.read_stata(f'{int_path}/panel_data_election_year.dta')
cols = ['ac_uq_id', 'unique_id', 'election_year', 
        'month_take', 'year_take', 'month_end', 
        'year_end']
panel_election_selcols = panel_election[ cols ].copy().drop_duplicates()
winners_info = winners[ [ 'unique_id', 'name', 'year' ] ] \
                        .rename( columns = { 'year' : 'election_year', 
                                            'name' : 'Politician_Name'} ) \
                        .merge(panel_election_selcols, on = ['unique_id', 'election_year'], how = 'left')

In [50]:
final_data = tweets_rubric1.merge(winners_info, on = ['ac_uq_id', 'Politician_Name'], how = 'left')

In [52]:
final_data.shape

(204360, 91)

In [56]:
204360-157500

46860

In [53]:
tweets_rubric1.shape

(157500, 85)

In [55]:
winners_info.shape

(2625, 8)

In [54]:
winners.shape

(2625, 14)

Index(['index', 'state', 'ac_uq_id', 'month', 'year', 'election_year',
       'month_take', 'year_take', 'month_end', 'year_end', 'ym', 'ym_take',
       'yeargov', 'constituency', 'acpost08ID', 'subfolder', 'unique_id',
       'age', 'ASSEMBLY', 'ASSEMBLY_1', 'DISTRICT', 'PARLIAMENT', 'P_NAME',
       'STATE_UT', 'dependent_1_owns_agricultural_as',
       'dependent_2_owns_agricultural_as', 'dependent_3_owns_agricultural_as',
       'self_owns_agricultural_assets', 'spouse_owns_agricultural_assets',
       'self_profession', 'self_profession_raw', 'spouse_profession_raw',
       'spouse_profession', 'education', 'state_clean', 'STATE_UT_clean'],
      dtype='object')

In [ ]:
winners[['unique_id', 'name']]

Index(['Unnamed: 0', 'unique_id', 'dependent_1_owns_agricultural_assets',
       'dependent_2_owns_agricultural_assets',
       'dependent_3_owns_agricultural_assets', 'self_owns_agricultural_assets',
       'spouse_owns_agricultural_assets', 'self_profession',
       'spouse_profession', 'name', 'ac_name', 'state', 'year', 'education'],
      dtype='object')